In [ ]:
import math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_optimizers import Momentum as CustomMomentum, NAG as CustomNAG

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_x_path = "data/train_x.csv"
train_y_path = "data/train_y.csv"
test_x_path  = "data/test_x.csv"
train_x = pd.read_csv(train_x_path, index_col=0)
train_y = pd.read_csv(train_y_path, index_col=0).iloc[:,0]  # 'year'
test_x  = pd.read_csv(test_x_path)
# test has 'id' column at the end; move it out
test_ids = test_x['id'].values
test_x = test_x.drop(columns=['id'])
print(train_x.shape, train_y.shape, test_x.shape)

(14000, 90) (14000,) (6000, 90)


In [12]:
# Split + scale
X_train, X_val, y_train, y_val = train_test_split(train_x.values, train_y.values, test_size=0.2, random_state=SEED)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(test_x.values)
D = X_train.shape[1]

# Target scaling (standardize year for training stability)
y_mean = float(y_train.mean())
y_std = float(y_train.std())
y_train_orig = y_train.copy()
y_val_orig = y_val.copy()
y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std

# For optional clipping at submission time
y_min = float(y_train_orig.min())
y_max = float(y_train_orig.max())

class ArrayDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X).float()
        self.y = None if y is None else torch.from_numpy(y).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        if self.y is None:
            return self.X[i]
        return self.X[i], self.y[i]

train_ds = ArrayDataset(X_train, y_train)
val_ds = ArrayDataset(X_val, y_val)
test_ds = ArrayDataset(X_test, None)

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

In [13]:
# MLP model for regression
class MLP(nn.Module):
    def __init__(self, d_in, hidden=[256, 128], dropout=0.1, bn=True):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            if bn: layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)

def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target)**2))

In [14]:
# Generic train/eval loop
def train_eval(run_name, optimizer_ctor, epochs=20, lr=3e-3, momentum=0.9):
    model = MLP(D, hidden=[512, 256, 128], dropout=0.2, bn=True).to(device)
    criterion = nn.MSELoss()
    if optimizer_ctor.__name__ in ('Momentum','NAG','CustomMomentum','CustomNAG'):
        optimizer = optimizer_ctor(model.parameters(), lr=lr, momentum=momentum)
    else:
        optimizer = optimizer_ctor(model.parameters(), lr=lr)
    best_val = float('inf'); best_state = None
    for epoch in range(1, epochs+1):
        model.train(); train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        # val
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        # Also report RMSE in original year units
        val_rmse = math.sqrt(val_loss) * y_std
        print(f"{run_name} | epoch {epoch:03d} | train MSE {train_loss:.4f} | val MSE {val_loss:.4f} | val RMSE(years) {val_rmse:.2f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().detach().clone() for k, v in model.state_dict().items()}
    # load best
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

In [15]:
# 1) Baseline Adam
adam_model, adam_val = train_eval('Adam', torch.optim.Adam, epochs=25, lr=1e-3)
# 2) Custom Momentum
m_model, m_val = train_eval('CustomMomentum', CustomMomentum, epochs=25, lr=5e-3, momentum=0.9)
# 3) Custom NAG
n_model, n_val = train_eval('CustomNAG', CustomNAG, epochs=25, lr=5e-3, momentum=0.9)

# Convert best validation MSE back to RMSE(years) for readability
results_mse = {'Adam': adam_val, 'Momentum': m_val, 'NAG': n_val}
results_rmse = {k: math.sqrt(v) * y_std for k, v in results_mse.items()}
print('Best val RMSE (years):', results_rmse)

Adam | epoch 001 | train MSE 0.8926 | val MSE 0.7207 | val RMSE(years) 9.41
Adam | epoch 002 | train MSE 0.7294 | val MSE 0.6950 | val RMSE(years) 9.24
Adam | epoch 002 | train MSE 0.7294 | val MSE 0.6950 | val RMSE(years) 9.24
Adam | epoch 003 | train MSE 0.6767 | val MSE 0.6981 | val RMSE(years) 9.26
Adam | epoch 003 | train MSE 0.6767 | val MSE 0.6981 | val RMSE(years) 9.26
Adam | epoch 004 | train MSE 0.6440 | val MSE 0.6870 | val RMSE(years) 9.18
Adam | epoch 004 | train MSE 0.6440 | val MSE 0.6870 | val RMSE(years) 9.18
Adam | epoch 005 | train MSE 0.6074 | val MSE 0.6904 | val RMSE(years) 9.21
Adam | epoch 005 | train MSE 0.6074 | val MSE 0.6904 | val RMSE(years) 9.21
Adam | epoch 006 | train MSE 0.5785 | val MSE 0.6871 | val RMSE(years) 9.18
Adam | epoch 006 | train MSE 0.5785 | val MSE 0.6871 | val RMSE(years) 9.18
Adam | epoch 007 | train MSE 0.5581 | val MSE 0.6836 | val RMSE(years) 9.16
Adam | epoch 007 | train MSE 0.5581 | val MSE 0.6836 | val RMSE(years) 9.16
Adam | epoch

In [16]:
# Choose best and generate predictions for submission
best_name, best_model = min((('Adam', adam_model), ('Momentum', m_model), ('NAG', n_model)), key=lambda kv: {'Adam': adam_val, 'Momentum': m_val, 'NAG': n_val}[kv[0]])
best_model = best_model.to(device).eval()

# Predict in standardized target space and invert to year units
preds_std = []
with torch.no_grad():
    for xb in test_loader:
        xb = xb.to(device)
        preds_std.append(best_model(xb).cpu().numpy())
preds_std = np.concatenate(preds_std).astype(np.float32)

# Invert scaling
pred_years = preds_std * y_std + y_mean
# Optional clipping to observed train range
pred_years = np.clip(pred_years, y_min, y_max)

sub = pd.DataFrame({'id': test_ids, 'year': pred_years})
sub_path = 'submission.csv'
sub.to_csv(sub_path, index=False)
print('Saved submission to', sub_path)
sub.head()

Saved submission to submission.csv


,id,year
0,3416,1999.176392
1,18991,2003.535645
2,11105,1997.772461
3,18902,2001.490967
4,18958,2000.764648


## Короткое описание решения
- Предобработка: train/val split (20%), стандартизация признаков по train, те же параметры на val/test.
- Модель: MLP c BatchNorm, ReLU, Dropout; метрика/лосс — MSE, отчёт RMSE можно восстановить как sqrt(MSE).
- Оптимизация: Adam (базовый), далее кастомные Momentum и NAG (PyTorch-реализация).
- печатаем лучшую валидацию для каждого оптимизатора.
- сохраняем submission.csv (id, year).

In [ ]:
import math
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR

class ResidualBlock(nn.Module):
    def __init__(self, d, dropout=0.1, norm='bn'):
        super().__init__()
        self.lin1 = nn.Linear(d, d)
        self.act = nn.SiLU()
        self.lin2 = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout) if dropout>0 else nn.Identity()
        if norm == 'bn':
            self.norm1 = nn.BatchNorm1d(d)
            self.norm2 = nn.BatchNorm1d(d)
        elif norm == 'ln':
            self.norm1 = nn.LayerNorm(d)
            self.norm2 = nn.LayerNorm(d)
        else:
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()

    def forward(self, x):
        residual = x
        out = self.norm1(x)
        out = self.lin1(out)
        out = self.act(out)
        out = self.dropout(out)
        out = self.norm2(out)
        out = self.lin2(out)
        out = self.dropout(out)
        return residual + out

class ResidualMLP(nn.Module):
    def __init__(self, d_in, widths=[512,512,256], blocks=1, dropout=0.1, norm='bn'):
        super().__init__()
        layers = []
        prev = d_in
        for w in widths:
            layers += [nn.Linear(prev, w), nn.SiLU()]
            for _ in range(blocks):
                layers += [ResidualBlock(w, dropout=dropout, norm=norm)]
            prev = w
        layers += [nn.LayerNorm(prev), nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)


def train_eval_(run_name, optimizer_ctor, epochs=40, lr=2e-3, momentum=0.9, weight_decay=1e-4, cosine=True, patience=8):
    model = ResidualMLP(D, widths=[768,512,256], blocks=1, dropout=0.15, norm='bn').to(device)
    criterion = nn.MSELoss()
    if optimizer_ctor.__name__ in ('Momentum','NAG','CustomMomentum','CustomNAG'):
        optimizer = optimizer_ctor(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        optimizer = optimizer_ctor(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs) if cosine else None

    best_val = float('inf'); best_state = None; bad = 0
    for epoch in range(1, epochs+1):
        model.train(); train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        # val
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        val_rmse = math.sqrt(val_loss) * y_std
        if scheduler: scheduler.step()
        print(f"{run_name}* | epoch {epoch:03d} | train MSE {train_loss:.4f} | val MSE {val_loss:.4f} | val RMSE {val_rmse:.2f}")
        if val_loss + 1e-7 < best_val:
            best_val = val_loss; best_state = {k: v.cpu().detach().clone() for k, v in model.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping."); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

In [ ]:
adamw_model, adamw_val = train_eval_('AdamW', torch.optim.AdamW, epochs=60, lr=2e-3, weight_decay=1e-4, cosine=True, patience=12)
nagw_model, nagw_val = train_eval_('CustomNAG-wd', CustomNAG, epochs=60, lr=3e-3, momentum=0.95, weight_decay=5e-5, cosine=True, patience=12)
print('Runs val RMSE (years):', { 'AdamW': math.sqrt(adamw_val) * y_std, 'CustomNAG-wd': math.sqrt(nagw_val) * y_std })

AdamW* | epoch 001 | train MSE 1.1994 | val MSE 0.7467 | val RMSE 9.57
AdamW* | epoch 002 | train MSE 0.7123 | val MSE 0.6967 | val RMSE 9.25
AdamW* | epoch 002 | train MSE 0.7123 | val MSE 0.6967 | val RMSE 9.25
AdamW* | epoch 003 | train MSE 0.6739 | val MSE 0.7015 | val RMSE 9.28
AdamW* | epoch 003 | train MSE 0.6739 | val MSE 0.7015 | val RMSE 9.28
AdamW* | epoch 004 | train MSE 0.6567 | val MSE 0.6872 | val RMSE 9.18
AdamW* | epoch 004 | train MSE 0.6567 | val MSE 0.6872 | val RMSE 9.18
AdamW* | epoch 005 | train MSE 0.6219 | val MSE 0.7265 | val RMSE 9.44
AdamW* | epoch 005 | train MSE 0.6219 | val MSE 0.7265 | val RMSE 9.44
AdamW* | epoch 006 | train MSE 0.6022 | val MSE 0.7058 | val RMSE 9.31
AdamW* | epoch 006 | train MSE 0.6022 | val MSE 0.7058 | val RMSE 9.31
AdamW* | epoch 007 | train MSE 0.5610 | val MSE 0.7102 | val RMSE 9.34
AdamW* | epoch 007 | train MSE 0.5610 | val MSE 0.7102 | val RMSE 9.34
AdamW* | epoch 008 | train MSE 0.5395 | val MSE 0.7298 | val RMSE 9.46
AdamW*

In [ ]:
# Choose best and generate predictions for submission
best_name, best_model = min((('AdamW', adamw_model), ('CustomNAG-wd', nagw_model)), key=lambda kv: {'AdamW': adamw_val, 'CustomNAG-wd': nagw_val}[kv[0]])
best_model = best_model.to(device).eval()
preds_std = []
with torch.no_grad():
    for xb in test_loader:
        xb = xb.to(device)
        preds_std.append(best_model(xb).cpu().numpy())
preds_std = np.concatenate(preds_std).astype(np.float32)
# Invert scaling
pred_years = preds_std * y_std + y_mean
# Optional clipping to observed train range
pred_years = np.clip(pred_years, y_min, y_max)
sub = pd.DataFrame({'id': test_ids, 'year': pred_years})
sub_path = 'submission1.csv'
sub.to_csv(sub_path, index=False)
print('Saved submission to', sub_path)

Saved strong submission to submission1.csv


- Residual-MLP c SiLU (Swish) и BatchNorm/LayerNorm.
- Регуляризация: weight decay (L2) и Dropout.
- Расписание: CosineAnnealingLR и early stopping по валидации.
- Сравнение: AdamW vs Custom NAG